In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()

In [ ]:
import cv2
import base64
from openai import OpenAI

# OpenAI API client
client = OpenAI()

# Function to encode image in Base64
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

# Extract frames from video
def extract_frames(video_path, num_frames=8):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    interval = max(1, total_frames // num_frames)
    
    frames = []
    for i in range(num_frames):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i * interval)
        ret, frame = cap.read()
        if ret:
            frame_path = f"frame_{i}.jpg"
            cv2.imwrite(frame_path, frame)
            frames.append(encode_image(frame_path))
    
    cap.release()
    return frames

# Video file and results below
#video_path = "1_ThrowIn_25.mp4"
#{
#  "Soccer Corner": 0.05,
#  "Soccer Free Kick": 0.10,
#  "Soccer Throw In": 0.85
#}

#video_path = "1_FreeKick_1.mp4"
#{
#  "Soccer Corner": 0.05,
#  "Soccer Free Kick": 0.90,
#  "Soccer Throw In": 0.05
#}

video_path = "1_CornerKick_3.mp4"
#{
#  "Soccer Corner": 0.05,
#  "Soccer Free Kick": 0.85,
#  "Soccer Throw In": 0.10
#}

frames_base64 = extract_frames(video_path, num_frames=8)

In [ ]:
from pydantic import BaseModel


class SoccerActionProbabilities(BaseModel):
    soccer_corner: float
    soccer_free_kick: float
    soccer_throw_in: float


# Prompt for classification
prompt = (
    "Classify the soccer action in the images. The images are frames from a video. "
    "Return probabilities for these exact fields: soccer_corner, soccer_free_kick, soccer_throw_in. "
    "Each probability must be a number between 0 and 1."
)

# Prepare request payload
input_content = [{"type": "input_text", "text": prompt}]
for frame_b64 in frames_base64:
    input_content.append(
        {
            "type": "input_image",
            "image_url": f"data:image/jpeg;base64,{frame_b64}",
        }
    )

# Send request to OpenAI using the Responses API with Pydantic parsing
response = client.responses.parse(
    model="gpt-4o-mini-2024-07-18",
    input=[{"role": "user", "content": input_content}],
    text_format=SoccerActionProbabilities,
)

# Print validated result
result = response.output_parsed
print(result.model_dump_json(indent=2))
